### Reimerei – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2022, 3. Runde, Junioraufgabe 1 (= 41. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 1)*

Aysun möchte für ihren kleinen Bruder Reimrätsel erstellen. Dafür benötigt sie Wörter, die sich reimen. Um mehr Auswahl zu
haben, lädt sie sich eine umfangreiche Liste von Wörtern aus dem Internet herunter. Nun braucht sie nur noch ein Programm,
das ihr aus dieser Liste passende Wortpaare heraussucht.

#### Junioraufgabe 1

Schreibe ein Programm, das eine Liste von Wörtern einliest und daraus alle passenden Wortpaare berechnet. Ein Wortpaar ist
passend, wenn es die folgenden Regeln erfüllt:

1. Die beiden Worte enden gleich: Sie haben dieselbe maßgebliche Vokalgruppe, und nach der maßgeblichen Vokalgruppe
   enthalten beide Wörter dieselben Buchstaben in derselben Reihenfolge. Dabei ist eine Vokalgruppe eine längstmögliche
   Folge von unmittelbar aufeinanderfolgenden Vokalen (z. B. hat das Wort *Taifun* die Vokalgruppen ‚ai‘ und ‚u‘), und die
   maßgebliche Vokalgruppe eines Wortes ist seine vorletzte Vokalgruppe, wenn das Wort zwei oder mehr Vokalgruppen enthält.
   Enthält ein Wort nur eine Vokalgruppe, ist seine maßgebliche Vokalgruppe die eine vorhandene Vokalgruppe.
2. In jedem der beiden Wörter enthält die maßgebliche Vokalgruppe und was ihr folgt mindestens die Hälfte der Buchstaben.
3. Keines der beiden Wörter darf mit dem kompletten anderen Wort enden.

Passende Wortpaare wären zum Beispiel *Baum, Traum* und *singen, klingen*; aber *Tanne, Rinne* verletzt Regel 1,
*Informatik, Akrobatik* verletzt Regel 2, und *kaufen, verkaufen* verletzt Regel 3.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `reimerei0.txt` bis `reimerei3.txt`, jeweils mit einem Wort pro Zeile.
`reimerei3.txt` ist mit 659 Wörtern die große Liste.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zeichenketten Zeichen für Zeichen durchgehen** und dabei Gruppen finden (hier: Vokalgruppen).
- **Alle Paare ausprobieren (Brute Force):** Mit zwei Schleifen ineinander betrachtet man jedes Paar von Wörtern genau einmal
  (die innere Schleife beginnt bei `i + 1`). Bei 659 Wörtern sind das etwa 217 000 Paare.
- **Vorberechnen:** Die maßgebliche Vokalgruppe eines Wortes brauchen wir für jedes Paar. Wir berechnen sie nur einmal pro Wort.
- **Regeln als Bedingungen:** Jede der drei Regeln wird eine kleine Prüfung. Ein Paar passt, wenn alle drei stimmen.
- **Gruppieren mit einem Wörterbuch (dict):** Statt alle Paare zu prüfen, sortiert man die Wörter nach ihrer Endung in Gruppen.
  Nur Wörter in derselben Gruppe können sich reimen. Das ist schneller (für Fortgeschrittene, am Ende des Notebooks).

**Nützliche Funktionen** (Bibliotheken braucht man keine)
- `wort.lower()`, `wort.strip()` (Leerzeichen am Rand entfernen), `wort[start:]` (Ausschnitt ab einer Stelle),
  `wort.endswith(ende)` (endet das Wort mit …?), `sorted(liste)`, `len`
- Ein Wörterbuch `{}` ordnet Schlüsseln Werte zu: `gruppen[schluessel] = [wort1, wort2]`

## Die Idee

Wir lesen alle Wörter, wandeln sie in Kleinbuchstaben um (damit „Biene“ und „biene“ gleich behandelt werden) und bestimmen für jedes Wort die
**maßgebliche Vokalgruppe**. Dann probieren wir jedes Paar aus und prüfen die drei Regeln.

**Vokale:** Wir zählen `a e i o u ä ö ü` als Vokale. Das `y` und `qu` zählen wir nicht als Vokale (siehe Hinweise unten).

**Vokalgruppen:** Beim Durchlaufen des Wortes merken wir uns, wo eine Vokalgruppe **beginnt**. Beim Wort „Taifun“ sind es `ai` (Start bei 1) und `u`
(Start bei 4). Die maßgebliche Vokalgruppe ist die vorletzte, also `ai`. Hat das Wort nur eine Vokalgruppe, ist es die einzige, und
bei keiner Vokalgruppe (zum Beispiel „LKW“) scheiden wir das Wort aus.

Sei `v` der Startindex der maßgeblichen Vokalgruppe im Wort. Dann ist `wort[v:]` das „Ende“: die maßgebliche Vokalgruppe mit allem, was folgt.

1. **Regel 1:** `wort1[v1:] == wort2[v2:]` (beide Enden sind gleich).
2. **Regel 2:** Das Ende hat mindestens die Hälfte der Buchstaben: `2 * len(wort[v:]) >= len(wort)`, für beide Wörter.
3. **Regel 3:** Weder `wort1.endswith(wort2)` noch `wort2.endswith(wort1)`.

## Schritt 1: Die Wörter einlesen

Eine Zeile pro Wort. `strip()` entfernt Leerzeichen und den Zeilenumbruch am Rand (in `reimerei2.txt` steht zum Beispiel hinter „Passfoto“ ein
Leerzeichen), `lower()` macht alles klein. Leere Zeilen werden übersprungen.

In [ ]:
def lies_woerter(dateiname):
    woerter = []
    with open(dateiname, encoding='utf-8') as f:
        for zeile in f:
            wort = zeile.strip().lower()
            if wort != '':
                woerter.append(wort)
    return woerter


woerter = lies_woerter('beispieldaten/reimerei0.txt')
print(woerter)

['bemühen', 'biene', 'breitschlagen', 'glühen', 'hersagen', 'hygiene', 'knecht', 'recht', 'schiene', 'schlank', 'schwank']


## Schritt 2: Die maßgebliche Vokalgruppe

`ist_vokal(buchstabe)` prüft, ob ein Buchstabe in `'aeiouäöü'` vorkommt.

Dann gehen wir das Wort durch. Eine Vokalgruppe **beginnt** an einer Stelle `i`, wenn dort ein Vokal steht und der Buchstabe davor kein Vokal
(oder `i` die erste Stelle ist). Wir sammeln die Startstellen aller Vokalgruppen in der Liste `starts`. Dann ist
`starts[-2]` die vorletzte Gruppe (wenn es mindestens zwei gibt) und `starts[0]` die einzige (wenn es nur eine gibt).
Hat das Wort keinen Vokal, geben wir `-1` zurück. Das bedeutet „es gibt keine maßgebliche Vokalgruppe“.

In [ ]:
VOKALE = 'aeiouäöü'


def ist_vokal(buchstabe):
    return buchstabe in VOKALE


def massgebliche_gruppe(wort):
    starts = []                             # Stellen, an denen eine Vokalgruppe beginnt
    for i in range(len(wort)):
        if ist_vokal(wort[i]) and (i == 0 or not ist_vokal(wort[i - 1])):
            starts.append(i)
    if len(starts) == 0:
        return -1                           # kein Vokal, z. B. 'lkw'
    if len(starts) == 1:
        return starts[0]
    return starts[-2]                       # die vorletzte Vokalgruppe


for wort in ['taifun', 'baum', 'singen', 'informatik', 'lkw', 's-bahn', 'hygiene']:
    v = massgebliche_gruppe(wort)
    print(f'{wort:12} Start der Gruppe: {v:2}', '  Ende:', wort[v:] if v >= 0 else '-')

taifun       Start der Gruppe:  1   Ende: aifun
baum         Start der Gruppe:  1   Ende: aum
singen       Start der Gruppe:  1   Ende: ingen
informatik   Start der Gruppe:  6   Ende: atik
lkw          Start der Gruppe: -1   Ende: -
s-bahn       Start der Gruppe:  3   Ende: ahn
hygiene      Start der Gruppe:  3   Ende: iene


Bei „Taifun“ ist es `aifun`, bei „Baum“ `aum`, bei „singen“ `ingen` (die Vokalgruppen sind `i` und `e`, die vorletzte ist `i`) und bei „Hygiene“ `iene`
(das `y` zählt nicht als Vokal, die Vokalgruppen sind `ie` und `e`, die vorletzte ist `ie`). Wörter wie „LKW“ haben keinen Vokal und werden ausgesondert.

## Schritt 3: Reimen sich zwei Wörter?

Die drei Regeln werden eine Funktion. Wir geben die Startstellen `v1` und `v2` der maßgeblichen Vokalgruppen mit, damit wir sie nicht für jedes Paar neu
berechnen müssen.

In [ ]:
def reimen_sich(wort1, v1, wort2, v2):
    ende1 = wort1[v1:]
    ende2 = wort2[v2:]
    # Regel 1: dieselbe maßgebliche Vokalgruppe und danach dieselben Buchstaben
    if ende1 != ende2:
        return False
    # Regel 2: das Ende hat mindestens die Hälfte der Buchstaben
    if 2 * len(ende1) < len(wort1) or 2 * len(ende2) < len(wort2):
        return False
    # Regel 3: keines der Wörter endet mit dem ganzen anderen Wort
    if wort1.endswith(wort2) or wort2.endswith(wort1):
        return False
    return True


# Die Beispiele aus der Aufgabenstellung
for a, b in [('baum', 'traum'), ('singen', 'klingen'), ('tanne', 'rinne'), ('informatik', 'akrobatik'), ('kaufen', 'verkaufen')]:
    print(a, b, reimen_sich(a, massgebliche_gruppe(a), b, massgebliche_gruppe(b)))

baum traum True
singen klingen True
tanne rinne False
informatik akrobatik False
kaufen verkaufen False


Die ersten beiden Paare passen (`True`). „Tanne, Rinne“ verletzt Regel 1 (`anne` ≠ `inne`), „Informatik, Akrobatik“ verletzt Regel 2
(das Ende `atik` hat nur 4 von 10 Buchstaben bei „informatik“), und „kaufen, verkaufen“ verletzt Regel 3.

## Schritt 4: Alle Paare suchen

Die Wörter, die keinen Vokal haben, werfen wir heraus. Dann berechnen wir für jedes Wort einmal die Startstelle `v` und probieren alle Paare `(i, j)` mit `i < j`
aus. Jedes Paar kommt so genau einmal vor. Damit die Ausgabe ordentlich aussieht, sortieren wir die beiden Wörter eines Paares alphabetisch
und sortieren am Ende die Liste der Paare. Auch doppelte Wörter in der Liste (zum Beispiel „Note“ zweimal) würden doppelte Paare erzeugen:
Ein Paar aus zwei gleichen Wörtern ist aber nie passend (Regel 3), und gleiche Paare würden nur einmal in die Ausgabe kommen.

In [ ]:
def finde_paare(woerter):
    # Wörter ohne Vokal aussortieren und die Startstelle merken
    liste = []
    for wort in woerter:
        v = massgebliche_gruppe(wort)
        if v >= 0:
            liste.append([wort, v])

    paare = []
    for i in range(len(liste)):
        for j in range(i + 1, len(liste)):
            wort1, v1 = liste[i]
            wort2, v2 = liste[j]
            if reimen_sich(wort1, v1, wort2, v2):
                paare.append(sorted([wort1, wort2]))
    return sorted(paare)


for nr in range(3):
    dateiname = f'beispieldaten/reimerei{nr}.txt'
    woerter = lies_woerter(dateiname)
    paare = finde_paare(woerter)
    print(f'{dateiname}: {len(woerter)} Wörter, Reimpaare: {len(paare)}')
    for a, b in paare:
        print(f'  {a} - {b}')

beispieldaten/reimerei0.txt: 11 Wörter, Reimpaare: 5
  bemühen - glühen
  biene - hygiene
  biene - schiene
  hygiene - schiene
  knecht - recht
beispieldaten/reimerei1.txt: 9 Wörter, Reimpaare: 2
  bildnis - wildnis
  brote - note
beispieldaten/reimerei2.txt: 6 Wörter, Reimpaare: 1
  epsilon - ypsilon


Das stimmt mit den Ergebnissen der offiziellen Lösungshinweise überein: `reimerei0.txt` hat fünf Reimpaare (zum Beispiel „knecht - recht“),
`reimerei1.txt` hat zwei und `reimerei2.txt` hat eins („epsilon - ypsilon“). Dass „Foto“ und „Passfoto“ sich nicht reimen, liegt an Regel 3.
„Poststempel“ und „Tempel“ ebenso.

## Schritt 5: Die große Liste

`reimerei3.txt` hat 659 Wörter. Das sind etwa 217 000 Paare, für den Computer ist das schnell. Wir geben die Paare in alphabetischer Reihenfolge aus,
in vier Spalten, damit die Liste nicht zu lang wird.

In [ ]:
woerter = lies_woerter('beispieldaten/reimerei3.txt')
paare = finde_paare(woerter)
print(f'reimerei3.txt: {len(woerter)} Wörter, Reimpaare: {len(paare)}')

zeilen = []
for a, b in paare:
    zeilen.append(f'{a} - {b}')
for i in range(0, len(zeilen), 4):
    spalten = []
    for text in zeilen[i:i + 4]:
        spalten.append(f'{text:28}')
    print(''.join(spalten))

reimerei3.txt: 659 Wörter, Reimpaare: 60
absender - kalender         ansage - frage              ansage - garage             bahn - zahn                 
bank - dank                 baum - raum                 bein - wein                 bier - tier                 
bild - schild               bitte - mitte               butter - großmutter         butter - mutter             
dame - name                 dezember - november         dezember - september        drucker - zucker            
durst - wurst               ermäßigung - kündigung      ermäßigung - reinigung      fest - test                 
feuer - steuer              fisch - tisch               flasche - tasche            frage - garage              
fuß - gruß                  gas - glas                  gleis - kreis               gleis - preis               
gleis - reis                glück - stück               gruppe - suppe              hand - land                 
hand - strand               hose - rose                

Es sind 60 Reimpaare. Die offiziellen Lösungshinweise nennen dazu noch das Paar „see - tee“: Es gehört nur zu einer anderen Version der Datei
(auf der Wettbewerbsseite gab es zwei Versionen). In der Version, die hier im Ordner liegt, kommt das Wort „Tee“ nicht vor, deshalb
fehlt das Paar. Alle anderen Paare stimmen mit den Hinweisen überein. Wörter mit Bindestrich (wie „s-bahn“) behandeln wir ganz normal,
der Bindestrich ist kein Vokal.

## Für Neugierige: Schneller mit Gruppen

Alle Paare zu probieren, wächst mit dem Quadrat der Wortanzahl. Bei einer Million Wörter wäre das viel zu langsam. Schneller geht es,
wenn man die Wörter **nach ihrem Ende gruppiert**: Nur Wörter mit demselben Ende `wort[v:]` können Regel 1 erfüllen.
Wir legen ein Wörterbuch an, in dem für jedes Ende alle Wörter mit diesem Ende gesammelt werden, und prüfen die Regeln 2 und 3 nur
innerhalb jeder Gruppe.

`gruppen.get(ende, [])` liefert die bisherige Liste für ein Ende oder eine leere Liste, wenn es das Ende noch nicht gibt.

In [ ]:
def finde_paare_schnell(woerter):
    gruppen = {}
    for wort in woerter:
        v = massgebliche_gruppe(wort)
        if v >= 0:
            ende = wort[v:]
            gruppen[ende] = gruppen.get(ende, []) + [[wort, v]]

    paare = []
    for ende in gruppen:
        gruppe = gruppen[ende]
        for i in range(len(gruppe)):
            for j in range(i + 1, len(gruppe)):
                wort1, v1 = gruppe[i]
                wort2, v2 = gruppe[j]
                if reimen_sich(wort1, v1, wort2, v2):
                    paare.append(sorted([wort1, wort2]))
    return sorted(paare)


for nr in range(4):
    woerter = lies_woerter(f'beispieldaten/reimerei{nr}.txt')
    print(f'reimerei{nr}.txt: beide Verfahren finden dieselben Paare: {finde_paare(woerter) == finde_paare_schnell(woerter)}')

reimerei0.txt: beide Verfahren finden dieselben Paare: True
reimerei1.txt: beide Verfahren finden dieselben Paare: True
reimerei2.txt: beide Verfahren finden dieselben Paare: True
reimerei3.txt: beide Verfahren finden dieselben Paare: True


**Sonderfälle und Entscheidungen**, die man in der Dokumentation nennen sollte:

- **Vokale:** `a e i o u ä ö ü`. Das `y` (in „Hygiene“, „Ypsilon“) und `qu` zählen wir nicht als Vokale.
- **Wörter ohne Vokal** (Abkürzungen wie „LKW“) werden ausgesondert.
- **Bindestriche** und andere Zeichen gelten als Nichtvokale. „s-bahn“ hat die Vokalgruppe `a`, das Ende ist `ahn`.
- **Groß- und Kleinschreibung** spielt keine Rolle.
- **Doppelte Wörter** in der Liste ergeben nie ein Reimpaar (Regel 3).